# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates loading, exploring, and processing the FAIR² dataset using the `mlcroissant` library, referencing all entities by their `@id`.

### Dataset Source
The dataset source is provided via a Croissant schema URL (FAIR² schema):

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Install mlcroissant if not already available
!pip install -q mlcroissant

## 1. Data Loading
Load and inspect the dataset's metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Version: {metadata.version}")
print(f"Published: {getattr(metadata, 'datePublished', 'N/A')}")
print(f"Authors: {getattr(metadata, 'author', 'N/A')}")


## 2. Data Overview
Review available record sets, fields, and their IDs (referenced by their `@id`).

In [ ]:
# List all available record sets and their fields by @id
print("Available record sets:")
for record_set in dataset.record_sets:
    print(f"  RecordSet @id: {record_set.id}")
    print(f"    Name: {record_set.name if hasattr(record_set, 'name') else ''}")
    print(f"    Description: {record_set.description if hasattr(record_set, 'description') else ''}")
    # List fields (columns) within each record set
    print("    Fields:")
    for field in record_set.fields:
        print(f"      Field @id: {field.id}; Name: {getattr(field, 'name', '')}; DataType: {getattr(field, 'data_type', '')}")
    print("-")

## 3. Data Extraction
Load all records from a specific record set (using the record set and field `@id`s as displayed above) into a DataFrame for analysis.

In [ ]:
# Let's collect all RecordSet @ids (usually 1 for a simple dataset)
record_set_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    dataframes[rs_id] = pd.DataFrame(records)

# For demonstration, pick the first record set's @id
main_record_set_id = record_set_ids[0]
print(f"Loaded columns in record set {main_record_set_id}:")
print(dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering, normalizing numeric fields, and grouping. Field selection is by field `@id` as shown in the overview above.

In [ ]:
# Get a list of numeric fields (float/integer) for the selected record set
from mlcroissant.types import DataType

rs_obj = None
for rs in dataset.record_sets:
    if rs.id == main_record_set_id:
        rs_obj = rs

numeric_fields = [field.id for field in rs_obj.fields if getattr(field, 'data_type', None) in [DataType.FLOAT, DataType.INTEGER]]
if numeric_fields:
    # Pick the first numeric field for analysis
    numeric_field_id = numeric_fields[0]
else:
    raise ValueError("No numeric fields found in the main record set.")

# Sample filtering: keep records where the numeric field > threshold
threshold = 10
col = numeric_field_id

# If the field @id is not the actual DataFrame column name, try also field.name
if col not in dataframes[main_record_set_id].columns:
    # Try replacing with field name
    field_obj = [f for f in rs_obj.fields if f.id == col][0]
    if hasattr(field_obj, 'name') and field_obj.name in dataframes[main_record_set_id].columns:
        col = field_obj.name
    else:
        col = dataframes[main_record_set_id].select_dtypes(include=['float','int']).columns[0]

filtered_df = dataframes[main_record_set_id][dataframes[main_record_set_id][col] > threshold]
print(f"Filtered records with {col} > {threshold}:")
print(filtered_df.head())

# Normalization
filtered_df[f"{col}_normalized"] = (filtered_df[col] - filtered_df[col].mean()) / filtered_df[col].std()
print(f"Normalized {col}:")
print(filtered_df[[col, f"{col}_normalized"]].head())

# Group by a text/categorical field (choose a non-numeric field)
cat_fields = [field.id for field in rs_obj.fields if getattr(field, 'data_type', None) not in [DataType.FLOAT, DataType.INTEGER]]
group_field_id = None
for f_id in cat_fields:
    candidate_field = f_id if f_id in filtered_df.columns else None
    if not candidate_field:
        # Try field 'name'
        f_objs = [f for f in rs_obj.fields if f.id == f_id]
        if f_objs and hasattr(f_objs[0], 'name') and f_objs[0].name in filtered_df.columns:
            candidate_field = f_objs[0].name
    if candidate_field:
        group_field_id = candidate_field
        break

if group_field_id:
    grouped_df = filtered_df.groupby(group_field_id)[col].mean()
    print(f"\nGrouped data by {group_field_id} (mean of {col}):")
    print(grouped_df.head())
else:
    print("No suitable categorical field found for grouping.")

## 5. Visualization
Visualize the numeric field's distribution (histogram) and, if available, a grouped boxplot by a categorical field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

# Histogram of the selected numeric field
plt.figure(figsize=(8,4))
filtered_df[col].hist(bins=30)
plt.title(f"Distribution of {col}")
plt.xlabel(col)
plt.ylabel('Count')
plt.show()

# If grouping field exists, plot boxplot
if group_field_id:
    plt.figure(figsize=(10,5))
    sns.boxplot(data=filtered_df, x=group_field_id, y=col)
    plt.title(f"Boxplot of {col} by {group_field_id}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
In this notebook, we loaded the FAIR² dataset, listed available record sets and fields using their `@id`, extracted and filtered tabular data, normalized a numeric field, grouped the results, and visualized key features. All references to dataset entities were made via their `@id`, ensuring schema consistency.

This workflow can be extended to further processing, automated ML, or integration with other FAIR-compliant datasets.